<a href="https://colab.research.google.com/github/pdf1802/f1-race-replay/blob/feature%2Fdata-analysis/analysis/notebooks/overtaking_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🏎️ Overtaking Probability Model

## What this notebook does
Builds a classification model that predicts the probability of an
overtake happening between two cars given race conditions.

This is a supervised classification problem :
- Input: race conditions between two cars
- Output: 1 = overtake happened, 0 = no overtake

## Why classification and not regression?
An overtake either happens or it doesn't — it's a discrete outcome.
We don't predict HOW MUCH an overtake happens, just WHETHER it does.
The model outputs a probability between 0 and 1.

## Features we'll use
- GapAhead: time gap to the car in front (seconds)
- TyreDelta: difference in tyre age between the two cars
- DRS: whether DRS is available (0 or 1)
- Compound: tyre compound of the car behind
- LapNumber: which lap of the race
- TrackTemp: track conditions

## What makes this hard
Overtakes are rare — most laps nobody passes anyone.
This is called class imbalance and we'll need to handle it carefully.

In [2]:
!pip install fastf1 pandas numpy matplotlib seaborn scikit-learn xgboost -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 135.5/135.5 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.6/69.6 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 40.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.6/55.6 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 3.5 MB/s eta 0:00:00


In [6]:
from google.colab import drive
drive.mount('/content/drive')

import os
import fastf1
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

os.makedirs('/content/drive/MyDrive/f1_cache',exist_ok=True)
fastf1.Cache.enable_cache('/content/drive/MyDrive/f1_cache')
print("Setup complete")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Setup complete


In [7]:
races = [
    (2023, 'Bahrain'), (2023, 'Saudi Arabia'), (2023, 'Australia'),
    (2023, 'Azerbaijan'), (2023, 'Miami'),
    (2024, 'Bahrain'), (2024, 'Saudi Arabia'), (2024, 'Australia'),
    (2024, 'Azerbaijan'), (2024, 'Miami'),
    (2025, 'Bahrain'), (2025, 'Saudi Arabia'), (2025, 'Australia'),
    (2025, 'Azerbaijan'), (2025, 'Miami'),
]

all_laps = []

for year, race in races:
    try:
        print(f"Loading {year} {race}...")
        session = fastf1.get_session(year, race, 'R')
        session.load(weather=True)
        laps = session.laps.copy()
        laps['Season'] = year
        laps['Race'] = race
        all_laps.append(laps)
        print(f"{year} {race} loaded")
    except Exception as e:
        print(f"{year} {race} failed: {e}")

full_data = pd.concat(all_laps, ignore_index=True)
print(f"\nTotal laps: {len(full_data)}")

Loading 2023 Bahrain...


core           INFO 	Loading data for Bahrain Grand Prix - Race [v3.8.1]
INFO:fastf1.fastf1.core:Loading data for Bahrain Grand Prix - Race [v3.8.1]
req            INFO 	No cached data found for session_info. Loading data...
INFO:fastf1.fastf1.req:No cached data found for session_info. Loading data...
_api           INFO 	Fetching session info data...
INFO:fastf1.api:Fetching session info data...
req            INFO 	Data has been written to cache!
INFO:fastf1.fastf1.req:Data has been written to cache!
req            INFO 	No cached data found for driver_info. Loading data...
INFO:fastf1.fastf1.req:No cached data found for driver_info. Loading data...
_api           INFO 	Fetching driver list...
INFO:fastf1.api:Fetching driver list...
req            INFO 	Data has been written to cache!
INFO:fastf1.fastf1.req:Data has been written to cache!
req            INFO 	No cached data found for session_status_data. Loading data...
INFO:fastf1.fastf1.req:No cached data found for session_status_d

2023 Bahrain loaded
Loading 2023 Saudi Arabia...


req            INFO 	Data has been written to cache!
INFO:fastf1.fastf1.req:Data has been written to cache!
req            INFO 	No cached data found for driver_info. Loading data...
INFO:fastf1.fastf1.req:No cached data found for driver_info. Loading data...
_api           INFO 	Fetching driver list...
INFO:fastf1.api:Fetching driver list...
req            INFO 	Data has been written to cache!
INFO:fastf1.fastf1.req:Data has been written to cache!
req            INFO 	No cached data found for session_status_data. Loading data...
INFO:fastf1.fastf1.req:No cached data found for session_status_data. Loading data...
_api           INFO 	Fetching session status data...
INFO:fastf1.api:Fetching session status data...
req            INFO 	Data has been written to cache!
INFO:fastf1.fastf1.req:Data has been written to cache!
req            INFO 	No cached data found for lap_count. Loading data...
INFO:fastf1.fastf1.req:No cached data found for lap_count. Loading data...
_api           INFO 	F

2023 Saudi Arabia loaded
Loading 2023 Australia...


...

2025 Australia loaded
Loading 2025 Azerbaijan...


req            INFO 	Data has been written to cache!
INFO:fastf1.fastf1.req:Data has been written to cache!
req            INFO 	No cached data found for driver_info. Loading data...
INFO:fastf1.fastf1.req:No cached data found for driver_info. Loading data...
_api           INFO 	Fetching driver list...
INFO:fastf1.api:Fetching driver list...
req            INFO 	Data has been written to cache!
INFO:fastf1.fastf1.req:Data has been written to cache!
req            INFO 	No cached data found for session_status_data. Loading data...
INFO:fastf1.fastf1.req:No cached data found for session_status_data. Loading data...
_api           INFO 	Fetching session status data...
INFO:fastf1.api:Fetching session status data...
req            INFO 	Data has been written to cache!
INFO:fastf1.fastf1.req:Data has been written to cache!
req            INFO 	No cached data found for lap_count. Loading data...
INFO:fastf1.fastf1.req:No cached data found for lap_count. Loading data...
_api           INFO 	F

2025 Azerbaijan loaded
Loading 2025 Miami...


req            INFO 	Data has been written to cache!
INFO:fastf1.fastf1.req:Data has been written to cache!
req            INFO 	No cached data found for driver_info. Loading data...
INFO:fastf1.fastf1.req:No cached data found for driver_info. Loading data...
_api           INFO 	Fetching driver list...
INFO:fastf1.api:Fetching driver list...
req            INFO 	Data has been written to cache!
INFO:fastf1.fastf1.req:Data has been written to cache!
req            INFO 	No cached data found for session_status_data. Loading data...
INFO:fastf1.fastf1.req:No cached data found for session_status_data. Loading data...
_api           INFO 	Fetching session status data...
INFO:fastf1.api:Fetching session status data...
req            INFO 	Data has been written to cache!
INFO:fastf1.fastf1.req:Data has been written to cache!
req            INFO 	No cached data found for lap_count. Loading data...
INFO:fastf1.fastf1.req:No cached data found for lap_count. Loading data...
_api           INFO 	F

2025 Miami loaded

Total laps: 15140


## Building the Overtaking Dataset

An overtake happens when a driver's position improves
from one lap to the next. We detect this by comparing
each driver's position lap by lap.

For each lap where a position change happened we also
need to know the conditions that made it possible:
- How big was the gap to the car ahead?
- Were the tyres fresher or older than the car ahead?
- Was DRS available?

This creates one row per lap per driver, with a label:
1 = this driver gained a position this lap (overtake)
0 = no position change

In [21]:
#Convert lap time to seconds
full_data['LapTimeSeconds'] = full_data['LapTime'].dt.total_seconds()

#Keep only what we need
laps= full_data[['Driver', 'Team', 'LapNumber', 'Position',
                  'Compound', 'TyreLife', 'LapTimeSeconds',
                  'Season', 'Race']].copy()


#Drop rows with missing position or lap time
laps=laps.dropna(subset=['Position','LapTimeSeconds','TyreLife'])

#Sort by race, driver , lap
laps = laps.sort_values(['Season', 'Race', 'Driver', 'LapNumber'])

#Detect position improvements(overtakes)
laps['PrevPosition']= laps.groupby(['Season', 'Race', 'Driver'])['Position'].shift(1)

laps['Overtake'] = (laps['Position']<laps['PrevPosition']) & (laps['PrevPosition'].notna()).astype(int)


print(f"Total laps: {len(laps)}")
print(f"Overtakes detected: {laps['Overtake'].sum()}")
print(f"Overtake rate: {laps['Overtake'].mean()*100:.1f}%")

Total laps: 14296
Overtakes detected: 1588
Overtake rate: 11.1%


## Adding Gap to Car Ahead

An overtake doesn't happen in isolation — it depends on
how close the car behind is to the car in front.

For each lap we calculate:
- Who is directly ahead of each driver
- What is the time gap between them
- What is the tyre age difference (fresher tyres = faster)

This gives us the context the model needs to understand
WHY an overtake happened or didn't.

In [22]:
# Build completely clean dataframe using .values to strip FastF1 completely
laps_clean = pd.DataFrame({
    'Driver': laps['Driver'].values,
    'Season': laps['Season'].values,
    'Race': laps['Race'].values,
    'LapNumber': pd.to_numeric(laps['LapNumber'], errors='coerce').values,
    'Position': pd.to_numeric(laps['Position'], errors='coerce').values,
    'Compound': laps['Compound'].values,
    'TyreLife': pd.to_numeric(laps['TyreLife'], errors='coerce').values,
    'LapTimeSec': pd.to_numeric(laps['LapTimeSeconds'], errors='coerce').values,
    'Overtake': laps['Overtake'].values,
})

# Sort by position within each lap of each race
laps_clean = laps_clean.sort_values(
    ['Season', 'Race', 'LapNumber', 'Position']
).reset_index(drop=True)

# Get car ahead using shift within each lap group
grp = laps_clean.groupby(['Season', 'Race', 'LapNumber'])
laps_clean['LapTimeAhead'] = grp['LapTimeSec'].shift(1)
laps_clean['TyreLifeAhead'] = grp['TyreLife'].shift(1)
laps_clean['CompoundAhead'] = grp['Compound'].shift(1)

# Calculate gap and tyre delta
laps_clean['GapAhead'] = (laps_clean['LapTimeSec'] - laps_clean['LapTimeAhead']).abs()
laps_clean['TyreDelta'] = laps_clean['TyreLife'] - laps_clean['TyreLifeAhead']

# Remove race leader (no car ahead)
laps_clean = laps_clean.dropna(subset=['TyreLifeAhead', 'GapAhead'])

print(f"Laps with car ahead data: {len(laps_clean)}")
print(f"Overtakes in this set: {laps_clean['Overtake'].sum()}")
print(f"\nSample:")
print(laps_clean[['Driver', 'LapNumber', 'Position',
                   'GapAhead', 'TyreDelta', 'Overtake']].head(10))

Laps with car ahead data: 13516
Overtakes in this set: 1558

Sample:
   Driver  LapNumber  Position  GapAhead  TyreDelta  Overtake
1     HAM        1.0       2.0     2.375        0.0     False
2     VER        1.0       3.0     1.342        0.0     False
3     SAI        1.0       4.0     2.295        0.0     False
4     ALO        1.0       5.0     1.677        1.0     False
5     ALB        1.0       6.0     0.830       -1.0     False
6     STR        1.0       7.0     1.679        1.0     False
7     GAS        1.0       8.0     2.241       -1.0     False
8     HUL        1.0       9.0     1.115        0.0     False
9     TSU        1.0      10.0     2.368        0.0     False
10    NOR        1.0      11.0     2.790        0.0     False
